#### **Accessing and Using the Benchmark Flood Inundation Mapping Database (FIMbench)**

This guide explains how to **access high-resolution benchmark Flood Inundation Maps (FIMs)** and **evaluate them using the FIMeval framework**, with clean integration into your existing workflows.
##### **Key resources**
- **FIMeval framework:** [[View the FIMeval repository]](https://github.com/sdmlua/fimeval)- This is a framewokr which provides an end-to-end interface for automated FIM evaluation, while also enabling access, retrieval, and reuse of benchmark FIM datasets directly within the evaluation modules.

- **BenchFIMQuery module:** [[Open the BenchFIMQuery module]](https://github.com/sdmlua/fimeval/tree/main/src/fimeval/BenchFIMQuery)- All the benchmark FIM query based on user defined filter and accessing those are written in this module. Users are welcome to explore original code if they need more flexibility.

- **Benchmark FIM database (S3 index):** [[Access the benchmark database]](https://sdmlab.ciroh.org/index.html#FIM_Database/)- All FIM tiers datasets with corresponding metadata for each event are stored within this S3 bucket.

##### **BenchFIMQuery**
For seamless evaluation with our own database, **BenchFIMQuery**, a module developed within **FIMeval** is used to interact with the benchmark datasets stored in an AWS S3 bucket. It supports querying, retrieving, and integrating benchmark FIM products directly into evaluation pipelines— without needing manual downloads and rearranging to run evaluation.

All benchmark data available and further usage notes are mentioned in fimbench viewer WebAPP: **https://fimbench.streamlit.app** 

**Import the FIMeval to use Query module**

In [1]:
#Once the environment is setup and fimeval is installed- for more info: https://github.com/sdmlua/fimeval?tab=readme-ov-file#3-set-up-virtual-environment

import fimeval as fe
from pathlib import Path

**Querying, accessing the benchmark FIM based on requirements**

In [ ]:
# User inputs: either model FIM raster, boundary AOI, or both
raster_path = "./paths/to/your/model_fim.tif"
boundary_path = "./paths/to/your/boundary.gpkg"

"""
Supports multiple combinations of filters. Choose ONE pattern and set the
others to None:

a) AOI-only search (raster or boundary), optional overlap stats.
b) AOI + exact date.
c) AOI + date range (with optional download).
d) Direct filename to download (no AOI/dates) – usually once you know
   the exact benchmark FIM name.
NOTE: if no date: returns all available benchmark FIMs for the AOI.

Common parameters
-----------------
raster_path:
    Optional path to user raster (e.g., model FIM).
boundary_path:
    Optional vector AOI file (can be used with or without raster).
huc8:
    Optional HUC8 filter (mainly for US basins).
event_date:
    Exact event date (optionally with hour).
start_date, end_date:
    Inclusive date range filter.
file_name:
    Exact benchmark FIM filename from the catalog.
area:
    If True and AOI given, return % overlap and km² vs benchmark AOI.
download:
    If True, download matched rasters/GPKGs to ``out_dir``.
out_dir:
    Directory for downloads (required if ``download=True``).
"""
#Use cases: User can choose one of the following patterns based on their needs

# a) AOI-only search (no dates, no filename)
log_aoi_only = fe.benchFIMquery(
    raster_path = raster_path,   # or None, if you only have boundary
    boundary_path = None,        # or boundary_path
    huc8 = None,
    event_date = None,
    start_date = None,
    end_date = None,
    file_name = None,
    area = True,                 # returns overlap stats vs benchmark AOI
    download = False,
    out_dir = None,
)
print("AOI-only search:", log_aoi_only)

# b) AOI + exact date
log_aoi_exact_date = fe.benchFIMquery(
    raster_path = raster_path,
    boundary_path = None,
    huc8 = None,
    event_date = "2017-05-01",   # YYYY-MM-DD or YYYY-MM-DD HH:MM
    start_date = None,
    end_date = None,
    file_name = None,
    area = True,
    download = False,
    out_dir = None,
)
print("AOI + exact date:", log_aoi_exact_date)

# c) AOI + date range (with optional download)
log_aoi_daterange = fe.benchFIMquery(
    raster_path = raster_path,
    boundary_path = None,
    huc8 = None,
    event_date = None,
    start_date = "2017-04-01",
    end_date = "2017-05-01",
    file_name = None,
    area = True,
    download = True,              # download all matches in this range
    out_dir = "./benchmark_downloads",
)
print("AOI + date range:", log_aoi_daterange)

# d) Direct filename download (no AOI, no dates)
log_by_filename = fe.benchFIMquery(
    raster_path = None,
    boundary_path = None,
    huc8 = None,
    event_date = None,
    start_date = None,
    end_date = None,
    file_name = "BENCHMARK_FIM_03020202_20170501.tif",  # example name
    area = False,               # ignored when no AOI is provided
    download = True,
    out_dir = "./benchmark_downloads",
)
print("Direct filename download:", log_by_filename)

**Run the Evaluation with the right benchmark**

In [ ]:
# Continuing from previous step, if user have intend to use FIM Evaluation Framework for evaluation
"""
01. Case Directory Structure
------------------------
The important understanding is for multi-case evaluation, the user need to have a main directory where each subfolder is a test case containing model FIMs to be evaluated.
For single case evaluation, user can provide the path to that single case folder.

Parameters
----------
Main_dir: root folder where each subfolder is a *test case*.
Example structure:
  Main_dir/
      HUC11110203_AR/
          model_fim_1.tif
          model_fim_2.tif
      HUC11110204_TX/
          model_fim_1.tif

For instance,
Main_dir = "path/to/your/Main_dir"

02. Positioning benchmark FIMs for evaluation
------------------------------
Now from Step 01, user will access the benchmark FIM for each case. Finding each case by running QUERY is precise way. However, the automation based on
area overlap, FIM tier and resolution priority is ongoing.

and Finally make a dictionary mapping each test case folder to the benchmark FIM filename obtained from Step 01. The FIM download in step 01, if done, does
not nessesarily place the benchmark FIMs in the case folders. So, User either provide the dictionary for each case by deciding the benchmark FIM filename from running
step 01 and getting them through log message OR download and map them based on folder- through right-out dir parameter in step01 this could be more automated.

benchmark_dict = {
    "HUC11110203_AR": "benchmark_01.tif"
    "HUC11110204_TX": "benchmark_02.tif"
}

03. Evaluation methods
------------------------------
While accessing the benchmark FIM, It will get all the benchmark boundary along with this, and use this boundary as `AOI` method for evaluation.

However, If user explicitly mention other methods like `smallest_extent` or `convex_hull`, FIMeval will use that method instead of benchmark AOI.

Evaluation methods:
"smallest_extent"  -> intersection of all FIM extents
"convex_hull"      -> convex hull around all FIM extents
"AOI"              -> use AOI shapefile as evaluation domain

method_name = "smallest_extent"  #for example

04. Other parameters
------------------------------
output_dir = "./path/to/output" # Optional: Directory to save evaluation results

Optional: user PWB (Permanent Water Bodies) dataset if not using the default one for the US.
PWB_dir = "./path/to/PWB"

target_crs = "EPSG:5070" # Optional: Target CRS (e.g., EPSG code or proj string, here for example is Albers Equal Area)

Optional: Target resolution in meters. If not provided, FIMeval can use the coarsest resolution among the inputs.
target_resolution = 10
"""

Main_dir = "path/to/your/Main_dir"  # Root folder containing case subfolders
benchmark_dict = {
    "HUC11110203_AR": "benchmark_01.tif",
    "HUC11110204_TX": "benchmark_02.tif",
}

output_dir = "./path/to/output"  # Optional: Directory to save evaluation results
method_name = "AOI"  # Example method name


# Evaluation usage examples
# Basic evaluation using default method & default PWB
fe.EvaluateFIM(
    Main_dir=Main_dir,
    benchmark_dict=benchmark_dict,
)

# Enforce target CRS / resolution and user AOI
fe.EvaluateFIM(
    Main_dir=Main_dir,
    method_name=None, #By default it is AOI--> boundary based on benchmark FIM, code automatically picks it up
    target_resolution= None,    
    target_crs=None,
    PWB_dir=None,       #If evaluating outside US or using custom PWB in boundary format
    output_dir=output_dir,
    benchmark_dict=benchmark_dict,
)

#Other results after evaluation

#For using default benchmark FIMs, method_name is AOI
#Print contingency maps (true/false positives, etc.)
fe.PrintContingencyMap(Main_dir, method_name, output_dir) 

#Plot evaluation metrics (CSI, POD, FAR, etc.)
fe.PlotEvaluationMetrics(Main_dir, method_name, output_dir)

#FIM evaluation with building footprints
countryISO = "US"  # e.g., "US" for United States
building_footprint = "./path/to/your/building_footprint.shp"

fe.EvaluationWithBuildingFootprint(
    Main_dir,
    method_name,
    output_dir,
    country=countryISO,
    geeprojectID="gee_project_id",  # GEE project ID containing building footprint dataset
)
#OR use local building footprint, using benchmark FIM, We are working on automating this step.
fe.EvaluationWithBuildingFootprint(
    Main_dir,
    method_name,
    output_dir,
    building_footprint=building_footprint,
)